# Adversarial-Attack Framework: Fast Gradient Sign Method (FGSM) and Projected Gradient Descent (PGD)
## Version 1: CICIoT2023 | All Training (Weighted) | Batch Size 128 | 10 Epochs | LR 0.05 | Seed 42 
## Target (y) = "label_multiclass" | Data Reupload = 2

## Setup

In [ ]:
%pip install -q pennylane pyarrow fastparquet

In [ ]:
import pennylane as qp
from pennylane import numpy as np
import torch
import torch.nn as nn

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix, classification_report

import sys
from pathlib import Path
import json
import time
from collections import Counter

In [ ]:
DATA_ROOT = Path("/kaggle/input/datasets/lawunnannda/quantum-sentinel-iot-v1-0")
data_path = f"{DATA_ROOT}/CICIoT2023/quantum"
target_col = "label_multiclass"

SCRIPTS_ROOT = Path("/kaggle/input/datasets/lawunnannda/quantum-sentinel-scripts")
sys.path.insert(0, str(SCRIPTS_ROOT))

log_dir = "/kaggle/working/logs"
Path(f"{log_dir}").mkdir(parents=True, exist_ok=True)

In [ ]:
from scripts.constants import DEFAULT_NOISE_RATE
from scripts.data import (
    class_balance_table,
    load_split,
    plot_class_balance_pie,
    plot_class_balance_bars,
    stratified_head,
)
from scripts.circuit import build_forward_circuit, create_quantum_device, initialize_weights
from scripts.loss import compute_l_ce
from scripts.logging import write_history_log
from scripts.utils import expectations_to_tensor, get_torch_device, to_np_x, to_torch_x

In [ ]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available?: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")
    print(f"Current CUDA device index: {torch.cuda.current_device()}")

## Load Dataset: CICIoT2023

In [ ]:
df = pd.read_parquet(f"{data_path}/q8_train.parquet")
df

In [ ]:
print(df["label_multiclass"].unique())
print()
print(df["label_family"].unique())

In [ ]:
# check rows if NaN dropped
df[df[target_col].notna()].copy()

In [ ]:
# get labels
class_names = sorted(df[target_col].dropna().unique())
class_names

In [ ]:
# load all data
X_train_full, y_train_full = load_split(data_path, "q8_train", target_col, class_names)
X_cal_full, y_cal_full = load_split(data_path, "q8_calibration", target_col, class_names)
X_test_full, y_test_full = load_split(data_path, "q8_test", target_col, class_names)
X_val_full, y_val_full = load_split(data_path, "q8_val", target_col, class_names)

In [ ]:
# check shape
[X_train_full.shape, y_train_full.shape], [X_cal_full.shape, y_cal_full.shape], [X_test_full.shape, y_test_full.shape], [X_val_full.shape, y_val_full.shape]

## EDA (Before Class-Weighting)

In [ ]:
# pie chart
splits = {
    "train": y_train_full,
    "calibration": y_cal_full,
    "test": y_test_full,
    "val": y_val_full,
}

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for ax, (name, y) in zip(axes.ravel(), splits.items()):
    plot_class_balance_pie(y, class_names, title=f"{name} (n={len(to_np_x(y))})", ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
plot_class_balance_bars(y_train_full, class_names, title="train")
plt.show()

## Class Balancing, Data Subset

- Rebalance **training only** (tempered class-weighted CE).
- Keep **val / calibration / test** unchanged (natural distribution).
- Do **not** use hard equal-count undersampling of every class.
- Do **not** use SMOTE after quantum encoding.
- Never train/tune on zero-day.

In [ ]:
# tempered class weights for train only: w_c ∝ 1/sqrt(count), clipped
y_np = to_np_x(y_train_full).astype(int)
counts = Counter(y_np.tolist())
raw = {c: 1.0 / (counts[c] ** 0.5) for c in counts}

# normalize to mean 1, then clip
mean_w = sum(raw.values()) / len(raw)
class_weight = {c: min(max(raw[c] / mean_w, 0.25), 4.0) for c in raw}

weight_tensor = torch.tensor(
    [class_weight.get(i, 1.0) for i in range(len(class_names))],
    dtype=torch.float32,
)

print("train class counts:", dict(sorted(counts.items())))
print()
print("tempered class weights:", {class_names[c]: round(class_weight[c], 3) for c in sorted(class_weight)})

In [ ]:
# uncomment to take subset
# from scripts.data import stratified_head
# X_train, y_train = stratified_head(X_train_full, y_train_full, 2000, seed=42)

X_train, y_train = X_train_full, y_train_full
X_cal, y_cal = X_cal_full, y_cal_full
X_test, y_test = X_test_full, y_test_full
X_val, y_val = X_val_full, y_val_full

# recompute tempered weights on the actual training y
y_np = to_np_x(y_train).astype(int)
counts = Counter(y_np.tolist())
raw = {c: 1.0 / (counts[c] ** 0.5) for c in counts}
mean_w = sum(raw.values()) / len(raw)
class_weight = {c: min(max(raw[c] / mean_w, 0.25), 4.0) for c in raw}
weight_tensor = torch.tensor(
    [class_weight.get(i, 1.0) for i in range(len(class_names))],
    dtype=torch.float32,
)

In [ ]:
# check shape
[X_train.shape, y_train.shape], [X_cal.shape, y_cal.shape], [X_test.shape, y_test.shape], [X_val.shape, y_val.shape]

## EDA (After Class-Weighting)

In [ ]:
num_classes = len(class_names)
num_qubits = X_train.shape[1]

print(f"classes: {class_names}")
print(f"\nnumber of classes: {num_classes}")
print(f"\nnumber of qubits: {num_qubits}")

In [ ]:
# initialize devices
dev = qp.device("default.mixed", wires=num_qubits)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# convert data type and move to device
X_train = to_torch_x(X_train, device)
y_train = to_torch_x(y_train, device).long()

# move weight tensor to device
weight_tensor = weight_tensor.to(device)

# verify
X_train[:5], y_train[:5], weight_tensor

In [ ]:
# check shape
[X_train.shape, y_train.shape], [X_cal.shape, y_cal.shape], [X_test.shape, y_test.shape], [X_val.shape, y_val.shape]

In [ ]:
# table
splits = {
    "train": y_train,
    "calibration": y_cal,
    "test": y_test,
    "val": y_val,
}

for name, y in splits.items():
    table = class_balance_table(y, class_names)
    print(f"{name} (n={len(to_np_x(y))})")
    display(table)

In [ ]:
# pie chart
splits = {
    "train": y_train,
    "calibration": y_cal,
    "test": y_test,
    "val": y_val,
}

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for ax, (name, y) in zip(axes.ravel(), splits.items()):
    plot_class_balance_pie(y, class_names, title=f"{name} (n={len(to_np_x(y))})", ax=ax)

plt.tight_layout()
plt.show()

## Quantum Circuit: Angle Encoding, Data Reuploading, and Adding Noise

$x\xrightarrow[\text{encode}]{\phi(x)}\ket{\psi(x)}\xrightarrow[\text{variational}]{U(\theta)}\ket{\Phi(x)}\xrightarrow{\Lambda_p}\rho(x)$
- $x$: classical data
- $\phi(x)$: `qp.AngleEmbedding()`
- $\ket{\psi(x)}$: quantum state after encoding
- $U(\theta)$: `qp.StronglyEntanglingLayers()`
- $\ket{\Phi(x)}$: quantum state after variational transform
- $\rho(x)=\Lambda_p(\ket{\Phi(x)}\bra{\Phi(x)})$: standard depolarization channel to model NISQ noise

In [ ]:
num_layers = 2                  # reupload same classical data
noise_rate = DEFAULT_NOISE_RATE

In [ ]:
# initialize device
dev = create_quantum_device(num_qubits)

# define circuit
forward_circuit = build_forward_circuit(dev, num_qubits, num_layers, noise_rate=noise_rate)

# initialize weights
theta = initialize_weights(num_layers, num_qubits, device)

# define classifier head
classifier_head = nn.Linear(num_qubits, num_classes).to(device)

In [ ]:
# initialize weights
theta = initialize_weights(num_layers, num_qubits, device)
theta.shape

In [ ]:
# DEBUG
theta

## Training

Training uses **cross-entropy only** (excluding $L_{intra}$ and $L_{inter}$).

$$
L = L_{CE} + \lambda_1 L_{intra} + \lambda_2 L_{inter}
$$

After setting $\lambda_1 = \lambda_2 = 0$,

$$
L = L_{CE}
$$

This is the baseline used to validate that FGSM/PGD degrade accuracy.

Minimizing $L_{CE}$ adjusts $\theta$ for better class predictions.

In [ ]:
lambda1 = 0.0   # no intra loss
lambda2 = 0.0   # no inter loss
epochs = 10
batch_size = 128
lr = 0.05
IMBALANCE_MODE = "class_weighted_ce"

In [ ]:
# define loss function
ce_loss_fn = nn.CrossEntropyLoss(weight=weight_tensor)
if IMBALANCE_MODE == "class_weighted_ce":
    ce_loss_fn = nn.CrossEntropyLoss(weight=weight_tensor)
elif IMBALANCE_MODE == "weighted_sampler":
    ce_loss_fn = nn.CrossEntropyLoss()
else:
    raise ValueError(f"unknown IMBALANCE_MODE={IMBALANCE_MODE}")

# define optimizer
optimizer = torch.optim.Adam(list([theta]) + list(classifier_head.parameters()), lr=lr)

# no PrototypeBank for plain VQC training

In [ ]:
def predict_labels(X, y, theta, classifier_head, forward_circuit, device):
    preds = []
    with torch.no_grad():
        for x in X:
            z, _ = forward_circuit(to_torch_x(x, device=device), theta)
            z = expectations_to_tensor(z)
            preds.append(int(classifier_head(z).argmax().item()))
    y_true = to_np_x(y).astype(int)
    return y_true, np.array(preds)

def ce_batch_loss(theta, classifier_head, ce_loss_fn, X_batch, y_batch, forward_circuit, device):
    # reuse CE term without MAQT
    return compute_l_ce(theta, classifier_head, ce_loss_fn, X_batch, y_batch, forward_circuit, device=device)

In [ ]:
history = {"loss": [], "l_ce": [], "val_macro_f1": [], "epoch_sec": []}

num_batches = (len(X_train) + batch_size - 1) // batch_size
print(f"plain VQC training: {len(X_train)} samples, {num_batches} batches/epoch, {epochs} epochs")

train_t0 = time.perf_counter()
for epoch in range(epochs):
    epoch_t0 = time.perf_counter()
    perm = torch.randperm(len(X_train), device=device)
    epoch_loss = []

    for batch_idx, start in enumerate(range(0, len(perm), batch_size)):
        idx = perm[start:start + batch_size]
        X_batch, y_batch = X_train[idx], y_train[idx]

        loss = ce_batch_loss(theta, classifier_head, ce_loss_fn, X_batch, y_batch, forward_circuit, device)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss.append(float(loss.item()))
        print(f"epoch {epoch+1} batch {batch_idx}/{num_batches-1}", flush=True)

    history["loss"].append(float(np.mean(epoch_loss)))
    history["l_ce"].append(history["loss"][-1])

    y_true, y_pred = predict_labels(X_val, y_val, theta, classifier_head, forward_circuit, device)
    val_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    history["val_macro_f1"].append(float(val_f1))
    history["epoch_sec"].append(time.perf_counter() - epoch_t0)

    print(
        f"epoch {epoch+1:02d}/{epochs} | loss={history['loss'][-1]:.4f} | "
        f"val_macro_f1={val_f1:.4f} | time={history['epoch_sec'][-1]:.1f}s"
    )

print(f"training done in {(time.perf_counter()-train_t0)/60:.1f} min")
theta_star = theta

In [ ]:
# write log (temporarily after training)
log_path = write_history_log(
    history,
    notebook="fgsm-pgd.ipynb",
    extra={
        "epochs": epochs,
        "batch_size": batch_size,
        "lr": lr,
        "lambda1": lambda1,
        "lambda2": lambda2,
    },
    log_dir="logs"
)
print(f"wrote {log_path.resolve()}")

In [ ]:
# DEBUG
theta_star

In [ ]:
# plots
epochs_x = range(1, len(history["loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_x, history["loss"], marker="o")
axes[0].set_title("Plain VQC CE loss (mean over batches / epoch)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel(r"$L = L_{CE}$")
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_x, history["val_macro_f1"], marker="o", color="tab:green")
axes[1].set_title("Validation macro-F1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("macro-F1")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluation on Known Test Samples

In [ ]:
label_ids = list(range(num_classes))

def predict_labels(X, y, theta, classifier_head, forward_circuit, device):
    preds = []
    with torch.no_grad():
        for x in X:
            z, _ = forward_circuit(to_torch_x(x, device=device), theta)
            z = z if torch.is_tensor(z) else torch.stack(z)
            preds.append(int(classifier_head(z.float()).argmax().item()))
    y_true = to_np_x(y).astype(int)
    return y_true, np.asarray(preds)

In [ ]:
y_true, y_pred = predict_labels(
    X_test, y_test, theta_star, classifier_head, forward_circuit, device
)

clean_acc = float(np.mean(y_true == y_pred))
clean_macro_f1 = float(
    f1_score(y_true, y_pred, average="macro", labels=label_ids, zero_division=0)
)

print(f"clean test accuracy: {clean_acc:.4f}")
print(f"clean test macro-F1: {clean_macro_f1:.4f}")
print(
    classification_report(
        y_true, y_pred,
        labels=label_ids,
        target_names=class_names,
        zero_division=0,
    )
)

cm = confusion_matrix(y_true, y_pred, labels=label_ids)
fig, ax = plt.subplots(figsize=(16, 10))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=np.arange(len(class_names)),
    yticks=np.arange(len(class_names)),
    xticklabels=class_names,
    yticklabels=class_names,
    xlabel="Predicted",
    ylabel="True",
    title="Plain VQC Test (CE only)",
)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
plt.tight_layout()
plt.show()

## Attack Helpers

In [ ]:
def ce_loss_on_x(x, y, theta, classifier_head, ce_loss_fn, forward_circuit, device):
    x = to_torch_x(x, device=device).detach().requires_grad_(True)
    y_t = torch.tensor([int(y)], device=device, dtype=torch.long)
    z, _ = forward_circuit(x, theta)
    logits = classifier_head(expectations_to_tensor(z)).unsqueeze(0)
    loss = ce_loss_fn(logits, y_t) if ce_loss_fn.weight is None else nn.CrossEntropyLoss()(logits, y_t)
    return loss, x

def fgsm_attack(x, y, theta, classifier_head, forward_circuit, eps, device, x_min=None, x_max=None):
    loss, x_var = ce_loss_on_x(x, y, theta, classifier_head, nn.CrossEntropyLoss(), forward_circuit, device)
    loss.backward()
    x_adv = x_var + eps * x_var.grad.sign()
    if x_min is not None: x_adv = torch.maximum(x_adv, x_min)
    if x_max is not None: x_adv = torch.minimum(x_adv, x_max)
    return x_adv.detach()

def pgd_attack(x, y, theta, classifier_head, forward_circuit, eps, alpha, steps, device, x_min=None, x_max=None):
    x0 = to_torch_x(x, device=device).detach()
    x_adv = x0.clone()
    for _ in range(steps):
        loss, x_var = ce_loss_on_x(x_adv, y, theta, classifier_head, nn.CrossEntropyLoss(), forward_circuit, device)
        loss.backward()
        x_adv = x_var + alpha * x_var.grad.sign()

        # project onto L_inf ball around x0
        x_adv = torch.maximum(torch.minimum(x_adv, x0 + eps), x0 - eps)
        if x_min is not None: x_adv = torch.maximum(x_adv, x_min)
        if x_max is not None: x_adv = torch.minimum(x_adv, x_max)
        x_adv = x_adv.detach()
    return x_adv

def eval_attacked(attack_fn, X, y, **kwargs):
    preds = []
    with torch.enable_grad():
        for x, label in zip(X, y):
            y_i = int(to_np_x(y).reshape(-1)[0])
            x_adv = attack_fn(x, y_i, theta_star, classifier_head, forward_circuit, device=device, **kwargs)

            with torch.no_grad():
                z, _ = forward_circuit(x_adv, theta_star)
                pred = int(classifier_head(expectations_to_tensor(z)).argmax().item())
            preds.append(pred)
    y_true = to_np_x(y).astype(int)
    preds = np.array(preds)
    return {
        "acc": float(np.mean(y_true == preds)),
        "macro_f1": float(f1_score(y_true, preds, average="macro", labels=label_ids, zero_division=0)),
    }

In [ ]:
EPS = 0.05
ALPHA = 0.01
PGD_STEPS = 10

In [ ]:
X_atk, y_atk = X_test, y_test

clean = {"acc": clean_acc, "macro_f1": clean_macro_f1}
fgsm = eval_attacked(lambda *a, **k: fgsm_attack(*a, eps=EPS, **k), X_atk, y_atk)
pgd = eval_attacked(lambda *a, **k: pgd_attack(*a, eps=EPS, alpha=ALPHA, steps=PGD_STEPS, **k), X_atk, y_atk)

print("clean :", clean)
print("FGSM  :", fgsm)
print("PGD   :", pgd)
print(f"FGSM drop (acc): {clean['acc'] - fgsm['acc']:+.4f}")
print(f"PGD  drop (acc): {clean['acc'] - pgd['acc']:+.4f}")
print(f"FGSM drop (macro-F1): {clean['macro_f1'] - fgsm['macro_f1']:+.4f}")
print(f"PGD  drop (macro-F1): {clean['macro_f1'] - pgd['macro_f1']:+.4f}")

if clean["acc"] <= 0:
    raise RuntimeError("clean acc is 0: fix data/training before attack check")

fgsm_hurt = (fgsm["acc"] < clean["acc"]) or (fgsm["macro_f1"] < clean["macro_f1"])
pgd_hurt = (pgd["acc"] < clean["acc"]) or (pgd["macro_f1"] < clean["macro_f1"])

if not fgsm_hurt:
    print("WARN: FGSM did not degrade acc/macro-F1 (try larger EPS)")
if not pgd_hurt:
    print("WARN: PGD did not degrade acc/macro-F1 (try larger EPS / more steps)")

if fgsm_hurt and pgd_hurt:
    print("OK: attacks degrade accuracy/F1 vs clean")

## Logging

In [ ]:
for k in history.keys():
    print(k)

In [ ]:
# write log (final)
log_path = write_history_log(
    history,
    notebook="fgsm-pgd.ipynb",
    extra={
        "label_column": "label_multiclass",
        "imbalance_mode": IMBALANCE_MODE,

        # plain VQC
        "epochs": epochs,
        "batch_size": batch_size,
        "lr": lr,
        "lambda1": lambda1,
        "lambda2": lambda2,
        "n_train": len(X_train),
        "n_val": len(X_val),
        "n_test": len(X_test),
        "clean_test_acc": clean_acc,
        "clean_test_macro_f1": clean_macro_f1,

        # attacks
        "fgsm_eps": EPS,
        "fgsm_test_acc": fgsm["acc"],
        "fgsm_test_macro_f1": fgsm["macro_f1"],
        "pgd_eps": EPS,
        "pgd_alpha": ALPHA,
        "pgd_steps": PGD_STEPS,
        "pgd_test_acc": pgd["acc"],
        "pgd_test_macro_f1": pgd["macro_f1"],

        "class_names": list(class_names),
    },
    log_dir="logs",
)
print(f"wrote {log_path.resolve()}")